In [1]:
#Важно! Хард код на Ямщикова, у него отсутсвия подряд, не перенесся дедлайн. Просрочек нет.
#25 июля до 16:00 рабочий день, в честь др Шорина

In [2]:
import pandas as pd
import numpy as np
import psycopg2
import tqdm
import datetime as dt
from datetime import datetime, timedelta,date
import openpyxl
from openpyxl.styles import NamedStyle, Font, Border, Side, Alignment, Color
from openpyxl import load_workbook
from openpyxl.utils.cell import get_column_letter
from openpyxl.comments import Comment
import smtplib 
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from email import encoders  
import os

In [3]:
#фукция для шапки таблицы
def set_style(ws, cell_range):
    thin = Side(border_style="thin", color="000000")
    for row in ws[cell_range]:
        for cell in row:
            cell.border = Border(top=thin, left=thin, right=thin, bottom=thin)
            cell.alignment = Alignment(vertical='top',horizontal ='center', wrap_text=True)
            cell.font = Font(name='Times New Roman', size=11, bold=True)

In [4]:
#стиль для таблицы
def set_table(ws, cell_range):
    thin = Side(border_style="thin", color="000000")
    for row in ws[cell_range]:
        for cell in row:
            cell.border = Border(top=thin, left=thin, right=thin, bottom=thin)
            cell.alignment = Alignment(horizontal='left', vertical ='center', wrap_text=True)
            cell.font = Font(name='Times New Roman', size=11)
            if 'не выполнен' in str(cell.value):
                    cell.font = Font(color="FF0000", bold=True)

### Поменять дату в письме(число, до которого можно писать по корректировкам.Сейчас стоит corr_date-текущая дата+3 дня), закоментить изменение почты и/или  руководителя

In [5]:
corr_date = date.today() + dt.timedelta(days=1)
print(corr_date)

2025-10-29


**Количество рабочих часов 528 (527.75)**


**Количество рабочих дней 66**

In [6]:
skud_va = psycopg2.connect (
  database="skud_va", 
  user="n_ilina", 
  password="Bkmbyf0317", 
  host="172.16.3.158", #"172.16.3.208", 
  port="5432"
)

Надо выкинуть из скрипта с kpi людей, кто устроился после запуска БК, а также Центральное представительво и новорогожку. 
Надо добавить время нахождения в офисе.

In [7]:
#выгрузка данных из бд
kpi_sql = """
with 
    calendar as (
    select 
        '2025-07-01'::date as sdate,
        '2025-09-30'::date as edate,
        sum(working_time )-1 as scheme_full  -- полное время за квартал по графику -- отняла один час, так как 25.07.2025 на час раньше отпустили из-за др Шорина
    from data_store.w_calendar wc
    where dt::date between '2025-07-01' and  '2025-09-30' and working_time >1
    ),
    work_sh as ( --рабочая схема для всех
        select 
            e.name  as name,
            case
	            when e.name like any (array['%Ившина %', '%Токаева Хеда %']) then count(tt.dt) filter (where tt.working_time>1)*4
            else round(sum(
                        tt.working_time),2) end as work_t_base,
            count(tt.dt) filter (where tt.working_time>1) as days_c_base
            from ( --новая таблица-календарь 
            select
                dt,
                working_time as working_time
                from data_store.w_calendar wc
                join calendar c on 1=1
                where dt between c.sdate and c.edate) tt
        cross join (select distinct
                        id,
                        concat (split_part(ce."name",' ',1),' ',split_part(ce."name",' ',2),' ',split_part(ce."name",' ',3)) as name,
                        min(start_dt) as start_dt
                    from data_store.c_employees ce 
                    --where ce.dismiss_dt isnull
                    where name not in ('Григорьев Игорь Владимирович', 'Колесников Вадим Сергеевич', 'Смолин Денис Викторович')

                    group by 1) ce
        join dds.employees e on e.tabel_number = ce.id 
        where dt>=ce.start_dt+ interval '1 days'
        	and ce.start_dt <= date '2025-07-01'
            and e.dept_id <> 21
        group by e.name, ce.name
        union
        select --схема для людей, учавствующих в статистике, но не согласующих удаленку
            name,
            case when name like 'Аракелян%' then count(tt.dt) filter (where tt.working_time>1)*4
            else sum (tt.working_time) end as scheme_time,
            count(tt.working_time) as count
        from dds.work_mart wm join
            (
            select
                dt,
                working_time
            from data_store.w_calendar wc
            join calendar c on 1=1 
                where dt between c.sdate and c.edate) tt
        on tt.dt = wm.date
        where name in ('Григорьев Игорь Владимирович', 'Колесников Вадим Сергеевич', 'Смолин Денис Викторович')

        group by name),
    task_time as (--списание времени
        select 
            employee as name,
            sum(work_time) as task_time,
            avg(anti_q) as avg_anti_q,
            Percentile_cont (0.5) WITHIN GROUP (ORDER BY anti_q) as median_anti_q
        from dds.tasktime_mart tm 
        join calendar c on 1=1
        where tm.created_date::date between c.sdate and c.edate
        group by 1),
 weeek_time_without_bitrix as (--списанеи времени в week
    select 
		employee,
		sum(tm."workTime") as weeek_task_time
	from dds.weeek_mart tm 
	join calendar c on 1=1
	left join dds.employees e on e.name = tm.employee
	where tm."writeOffDate"::DATE between c.sdate and c.edate
group by 1
),

-- bitrix_april AS (
--				SELECT employee, sum(work_time) AS sum_worktime_bitrix
--				FROM dds.tasktime_mart 
--				WHERE created_date BETWEEN '2025-04-01' AND '2025-04-30'
--				group BY employee
--				),

weeek_time AS (
				SELECT wtwb.employee AS employee, weeek_task_time
				FROM weeek_time_without_bitrix  AS wtwb
				),
absence as ( --отсутствия для расчета времени
        select 
            name,
            coalesce (sum(abs_time) filter (where info like any (array['%омандировка%','%стреча%','%о поручению руководителя%'])),0) as bt_time,
            coalesce (sum(abs_time) filter (where info like any (array['%стреча%','%о поручению руководителя%'])),0) as task_bt_time,--командировки, встречи и поручения не попадают во время списания
            case when name ='Дряев Георгий Георгиевич' then 15.5 else coalesce (sum(abs_time) filter (where info like any (array['%даленная работа%'])),0) end as work_abs,
            coalesce  (sum(abs_time) filter (where info like any (array['другое','%о состоянию здоровья%'])),0) as work_not_task_abs,
            coalesce (sum(abs_time) filter (where info not like all (array['%омандировка%','%даленная работа%','%стреча%','%о поручению руководителя%','%по состоянию здоровья%','другое','болезнь без оплаты', 'дополнительные выходные дни не оплачиваемые', 
                                        'отпуск неоплачиваемый по разрешению работодателя'])),0) as vac_time,--болезни и отпуска
            coalesce (sum(abs_time) filter (where info like any (array['болезнь без оплаты', 'дополнительные выходные дни не оплачиваемые', 
                                        'отпуск неоплачиваемый по разрешению работодателя'])),0) as vac_not_pay
        from(--выбирается последняя информация типа отсутствия
        select 
            distinct
            dates,
            name,
            no_stat,
            EXTRACT(DOW FROM dates) as week_day,
            least (
            sum(extract(epoch from --берем меньшее из суммы отсутствий и продолжительности рабочего дня
            case
                when info ='командировка' then dates::date+ end_wd::time
            when (end_date::time = '00:00:00'::time or end_date isnull) and end_wd - start_wd > interval '4 hour' then dates::date + (end_wd)::time --- (9 - working_time ) *interval '1 hour'+interval '45 minute')::time
            when (end_date::time = '00:00:00'::time or end_date isnull) and end_wd - start_wd <= interval '4 hour' then dates::date +end_wd
            when (end_date::time = '00:00:00'::time or end_date isnull) then dates::date + coalesce(end_wd::time,'18:00:00'::time)
            else end_date
            end -
            case 
                when start_date isnull then dates::date+coalesce(start_wd::time, '09:00:00'::time)
                when start_date::date = end_date::date and start_date::time<>'00:00:00' then start_date
                when start_date::date = end_date::date
                    and start_date::time = '00:00:00'
                    then start_date::date + start_wd::time
                when start_date::date <= dates then dates + start_wd
                else start_date  
            end
            )/3600),wc.working_time, 8.25) as abs_time,
            lower(min(info)) as info
        from 
        (select 
            GENERATE_SERIES(start_date::date,end_date::date, INTERVAL '1 days')::date  as  dates,
            ce.name,
            no_stat,
            start_date,
            ca.end_date,
            end_wd,
            start_wd,
            ca.created_date,
            ca.typeofabsence ,
            coalesce (bat.info, ca.typeofabsence) as info,
            case when ca.created_date > ca.start_date then 1 else 0 end as last_fill	
        from (select --выборка с учетом некорректного заполения отсутствия до утра следующего дня
                cab.cardnumber,
                cab.typeofabsence,
                case when cab.start_date::time> cab.end_date::time and end_date::time <>'00:00:00'::time then cab.end_date else cab.start_date end as start_date,
                created_date,
                case 
                    when cab.start_date::time> cab.end_date::time and end_date::time <>'00:00:00'::time then cab.start_date 
                    when length(cab.typeofabsence)<3 and cab.start_date::date <> cab.end_date::date then cab.start_date::date +'00:00:00'::time else cab.end_date end as end_date
                from data_store.c_absences cab
                join calendar c on 1=1
                where start_date::date between c.sdate and c.edate or 
                    end_date between  c.sdate and c.edate ) ca
        join data_store.c_employees ce on ce.id = ca.cardnumber 
        join calendar c on 1=1
        left join dds.employees e on e.tabel_number = ca.cardnumber
        left join data_store.b_absence_type bat on bat.typeofabsence = ca.typeofabsence 
        ) pa
        join ( select dt,
                working_time as working_time
                from data_store.w_calendar wc1
                join calendar c on 1=1
                where dt::date between  c.sdate and c.edate) wc on wc.dt = pa.dates
        where info not in ('Отсутствие по невыясненным причинам') and (wc.working_time > 1 or pa.typeofabsence ='Командировка' )
        group by 1,2,3,4, working_time,end_date,start_date,end_wd,start_wd)ab
        group by 1),
    work_time_with_dinner as (
    select 
            wm.name,
            e.no_stat,
            round(sum (case 
	            	when wm.name like 'Аракелян%' then extract(epoch from end_time -start_time) else extract(epoch from wm.working_time) end )/3600,2) as with_dinner
        from dds.work_mart wm 
        join dds.employees e on e."name" = wm.name
        join data_store.c_employees ce on ce.id = e.tabel_number
        join calendar c on 1=1
        where date between c.sdate and c.edate and date>ce.start_dt --убираем  первый день
       -- where date between '2025-04-01' and '2025-04-11' and date>ce.start_dt --убираем  первый день
        group by e.tabel_number,wm.name, e.no_stat)
    ,
    skid_counts as (
    SELECT
				e."name",
				round (extract (hour from (sum (wm.working_time)+interval '30 minutes')), 0) as sum_working_time,
				COUNT (case when stat = true then late_time end) AS late_cnt,
								COUNT (case when stat = true then 1 end) AS all_cnt,
				round (round(extract (epoch from sum (wm.working_time)/3600),2) / 
				round((case 
                    when e.name='Аракелян Карен Эдуардович' and sum(wm.wt)>1 then sum(wm.wt)*1800
                    when sum(wm.wt)>1 then sum(wm.wt)*3600 else sum(wm.wt) end/3600),2) *100, 2) AS avg_working_time,
				extract (epoch from sum (wm.working_time)),
				case when sum(wm.wt)>1 then sum(wm.wt)*3600 else sum(wm.wt) end
		FROM dds.employees e 
		left join (select wm.*,
		case when wm.name ='Николаев Сергей Юрьевич' and wm.date = '2024-11-27' then 0.000001 else wc.working_time end as wt
				from 
				dds.work_mart wm 
				join calendar c on 1=1
				join data_store.w_calendar wc on wc.dt =wm.date
				WHERE
						"date" between c.sdate and c.edate
						and not (wm.b_abc isnull and start_time='09:00:00'::time) ) wm on e.name = wm.name
		GROUP BY e."name"
    ),
    late_min as (
        select --опоздания
            e.name,
            sum(case when wm.name in ('Ананович Владимир Борисович','Тигиев Заур Эрикович','Четверов Владимир Валерьевич',
            'Селезнева Марианна Валерьевна','Осокина Наталья Геннадьевна','Токарева Наталья Геннадьевна',
            'Курбатова Марина Александровна','Силенков Владимир Павлович','Яковлева Екатерина Александровна',
            'Аракелян Карен Эдуардович','Кореницын Павел Николаевич','Кораблев Дмитрий Валентинович',
            'Громова Любовь Андреевна','Шумова Вера Васильевна','Суздальцев Алексей Васильевич','Шубин Геннадий Николаевич',
            'Рыльская Анжелика Анатольевна','Кириллова Елена Владиславовна'
            ) and wm.date >'2024-11-20' then 0 
            when wm.name ='Григорьев Игорь Владимирович' and wm.date ='2025-01-14' then 0
            when wm.name ='Трубникова Наталья Николаевна' and wm.date >'2025-03-19' then 0
            when wm.name ='Румянцева Анастасия Игоревна' and wm.date ='2025-04-02' then 0
            --when wm.name = 'Сребдольская Екатерина Петровна' then extract(epoch from wm.start_time -'09:00:00'::time)
            else
            extract(epoch from wm.start_time -e.start_wd) end)/60 as late_min
        from dds.work_mart wm 
        join dds.employees e on wm.name = e.name 
        join data_store.c_employees ce on ce.id  = e.tabel_number 
        join (select *
                from
                data_store.w_calendar
                where working_time>1) wc on wc.dt = wm."date" 
        join calendar c on 1=1
        where wm.date between c.sdate and c.edate and wm.date not between '2024-11-20' and '2024-11-30' 
        and wm.start_time >e.start_wd and stat =TRue
        --(case when wm.name = 'Сребдольская Екатерина Петровна' then '09:00:00'::time else e.start_wd end)
        and wm.date>ce.start_dt+interval '1 days' 
        group by e.name),
    empls as (
    select
        e."name" ,
        e.dept_id ,
        e.str_lvl ,
        d.short_name as dept,
        be.email as email,
        e.deleted,
        case 
	       when d.short_name in ('Направление аналитики больших данных','Департамент инноваций и инжиниринга','Центр инжиниринга'
	        ,'Направление инноваций') then 'Ульянов Евгений Ильич'
	        when d.short_name in ('Направление ИБ') then 'Шамко Дмитрий Борисович'
	        when d.short_name in ('Центр мониторинга и анализа качества','Направление проверки и оценки предприятий','Направление стандартизации') then 'Афанасьева Маргарита Александровна'
	        when e.name like 'Чубукова%' then 'Кошелева Елизавета Робертовна'
	        when d.short_name in ('Секретариат','ДОКР','Советник ГД') then 'Волкова Наталья Васильевна' 
            when e.name like any (array['Каменскова%','Шевяков%']) then 'Волкова Наталья Васильевна'
            when e.name like 'Пшеничный%' then 'Пшеничный Дмитрий Алексеевич' 
            when e.name like 'Тигиев%' then 'Виноградов Всеволод Владимирович' 
            when e.name like any (array['Трусов Михаил%','Краснова Светлана Николаевна','Ефремов Василий Петрович']) then 'Ефремов Василий Петрович' 
            when d.short_name in ('Направление по продажам') then 'Барашкина Екатерина Сергеевна'
            when e.name like any (array['Мазур%','Яковлева Елена%']) then 'Мазур Алексей Борисович' 
            when e.name like 'Клишин%' then 'Клишин Андрей Иванович'
            when d.short_name ='ДУП' then 'Радченко Наталья Владимировна'
            -- when d.short_name IN ('Департамент цифровизации','Центр разработки','Направление продаж') then 'Кондратенко Денис Анатольевич'
            -- when d.short_name ='Центр сертификации' then 'Грудцов Иван Александрович'
            -- when d.short_name IN ('АО "Кволити +"','Орган сертификации систем менеджмента','Орган сертификации продуктов') then 'Ефремов Василий Петрович'
            -- when d.short_name IN ('Центральное Тех Представительство','Департамент оценки и технического контроля','Департамент оценки и ТК') then 'Ткачев Сергей Владимирович'
            -- when d.short_name ='Департамент продаж' then 'Моисеенко Виталий Евгеньевич'
            end as head,
        case 
	   when d.short_name in ('Направление аналитики больших данных','Департамент инноваций и инжиниринга','Центр инжиниринга', 'Направление инноваций') 
	    then 'e.ulyanov@rt-techpriemka.ru'  
	    when d.short_name in ('Направление ИБ') then 'd.shamko@rt-techpriemka.ru'
	    when d.short_name in ('Центр мониторинга и анализа качества','Направление проверки и оценки предприятий','Направление стандартизации') then 'm.afanasyeva@rt-techpriemka.ru'
        when e.name = 'Чубукова Александра Сергеевна' then 'e.kosheleva@rt-techpriemka.ru'
        when d.short_name in ('Секретариат','ДОКР','Советник ГД') then 'n.volkova@rt-techpriemka.ru'
        when e.name like any (array['Каменскова%','Шевяков%']) then 'n.volkova@rt-techpriemka.ru'
        when e.name like 'Пшеничный%' then 'd.pshenichnyy@rt-techpriemka.ru' 
        when e.name like 'Тигиев%' then 'v.vinogradov@rt-techpriemka.ru'
        when d.short_name in ('Направление по продажам') then 'e.barashkina@rtqualityplus.ru'
        when e.name like any (array['Трусов Михаил%','Краснова Светлана Николаевна', 'Ефремов Василий Петрович']) then 'v.efremov@rtqualityplus.ru'
        when e.name like any (array['Мазур%','Яковлева Елена%']) then 'a.mazur@rt-techpriemka.ru'
        when e.name like 'Клишин%' then 'a.klishin@rt-techpriemka.ru'
        when d.short_name ='ДУП' then 'n.radchenko@rt-techpriemka.ru'
        -- when d.short_name IN ('Департамент цифровизации','Центр разработки','Направление продаж') then 'd.kondratenko@rt-techpriemka.ru'
        -- when d.short_name ='Центр сертификации' then 'i.grudcov@rt-techpriemka.ru'
        -- when d.short_name  IN ('АО "Кволити +"','Орган сертификации систем менеджмента','Орган сертификации продуктов') then 'v.efremov@rtqualityplus.ru'
        -- when d.short_name IN ('Центральное Тех Представительство','Департамент оценки и технического контроля','Департамент оценки и ТК') then 's.tkachev@rt-techpriemka.ru'
        -- when d.short_name ='Департамент продаж' then 'v.moiseenko@rt-techpriemka.ru'
        end	as head_mail 
    from dds.employees e
    left join data_store.c_employees ce on ce.uniq_id = e.uniq_id and ce.deleted = false
    join dds.departments d on d.dept_id = e.dept_id
    join data_store.b_employees be on be.name = e.name 
    --where (str_lvl not in (1,0))
    order by d.short_name , e.str_lvl
    ),
    head_empls as (
    select 
        distinct
        coalesce(e.head,h.name) as head_name,
        coalesce (e.head_mail,h.email) as email,
        e.email as emp_mail,
        e.name as name
    from empls e  
    left join empls h on h.dept = e.dept 
    where (e.str_lvl in (2,3) and h.str_lvl =2 and h.deleted <>True) 
    or e.head like any (array ['Шамко%','%Волкова%',
    'Ульянов%','Пшеничный%','Старшинов%','Мазур%','Барашкина%','Виноградов%','Клишин%','Кошелева%','Ефремов%'])
    ),
    pivot as (
    select  
        c.scheme_full,
        ws.name as employee,
        be.email as emp_mail,
        e2."name" as zgd,
        e2.login||'@rt-techpriemka.ru' as zgd_email,
        e.str_lvl,
        coalesce (wt.no_stat, e.no_stat) as no_stat,
        d.short_name as dept,
        days_c_base,
        work_t_base,
        with_dinner,
        with_dinner- coalesce (vac_time,0) as working_with_abs,
        with_dinner- coalesce (vac_time,0)- coalesce (vac_not_pay,0) as work_time,
        work_t_base- coalesce (vac_time,0)- coalesce (vac_not_pay,0) - coalesce (work_not_task_abs,0) -coalesce (bt_time,0) as time_for_task,
        round((work_t_base-coalesce (vac_time,0)-coalesce (vac_not_pay,0))::numeric,2) as time_for_work,
        round(work_t_base- coalesce (vac_time,0)-coalesce (vac_not_pay,0)-coalesce (bt_time,0)::numeric,2) as time_in_office,
        case when with_dinner-coalesce (vac_time,0)>=0 
        then round((with_dinner-coalesce (vac_time,0))/
            (work_t_base-coalesce (vac_time,0)-coalesce (vac_not_pay,0))*100,2)
        --else round((with_dinner)/(work_t_base-coalesce (vac_time,0)-coalesce (vac_not_pay,0))*100,2) 
        end as pers_work_dinner,
        sc.avg_working_time,--процент отработанного времени от рабочей схемы за вычетом отпусков и отгулов
        round(coalesce (vac_time,0),2) as vac_time,
        round(coalesce (vac_not_pay,0),2) as vac_not_pay,
        round(coalesce (bt_time,0),2) as bt_time,
        round(coalesce (work_abs,0),2) as work_abs,
        round(coalesce (work_not_task_abs,0),2) as work_not_task_abs,
        round(coalesce (task_bt_time,0),2) as task_bt_abs,
        coalesce(round(late_min,1),0) as late_min,
        case when e2.name ='Леонов Александр Владимирович' then coalesce(round(weeek_task_time::numeric,2),0) else 
        coalesce(round(task_time::numeric,2),0) end as task_time,
        coalesce (case 
	        when ws.name ='Дряев Георгий Георгиевич' then
	        round(task_time::numeric/((work_t_base-coalesce (vac_time,0)
                -coalesce (work_not_task_abs,0)-coalesce(bt_time,0)-coalesce (vac_not_pay,0)+48)::numeric)*100,2)
	        when ws.name ='Токарева Наталья Геннадьевна' then round(task_time::numeric/((scheme_full-coalesce(bt_time,0)-coalesce (vac_time,0)
                -coalesce (work_not_task_abs,0)-coalesce (vac_not_pay,0))::numeric)*100,2)
            when ws.name in ('Лебедев Александр Владимирович','Березовская Анастасия Владимировна')
            then round(task_time::numeric/((work_t_base-coalesce (vac_time,0)
                -coalesce (work_not_task_abs,0)-coalesce (vac_not_pay,0))::numeric)*100,2)
            when (ws.name  ='Григорьев Игорь Владимирович') 
            then round(((weeek_task_time/(c.scheme_full-coalesce(bt_time,0)- coalesce (vac_time,0)-coalesce (vac_not_pay,0)))::numeric)*100,2)
            when (e2.name ='Леонов Александр Владимирович' AND ws.name!='Клячкина Екатерина Александровна') 
            then round(weeek_task_time::numeric/((work_t_base-coalesce (vac_time,0)
                -coalesce (work_not_task_abs,0)-coalesce(bt_time,0)-coalesce (vac_not_pay,0))::numeric)*100,2)
            when (ws.name in ('Колесников Вадим Сергеевич', 'Смолин Денис Викторович', 'Четверов Владимир Валерьевич')) 
               then round(((task_time/(c.scheme_full-coalesce(bt_time,0)- coalesce (vac_time,0)-coalesce (vac_not_pay,0)))::numeric)*100,2) 
            else 
            round(task_time::numeric/((work_t_base-coalesce (vac_time,0)
                -coalesce (work_not_task_abs,0)-coalesce(bt_time,0)-coalesce (vac_not_pay,0))::numeric)*100,2) end,0) as pers_task,
        he.head_name,
        he.email
    from work_sh ws 
    left join dds.employees e on e.name = ws.name
    left join dds.departments d on d.dept_id = e.dept_id
    left join dds.departments d1 on d1.short_name = d.parent_dept 
    left join dds.employees e2 on e2.dept_id = d1.dept_id 
    left join work_time_with_dinner wt on wt.name = ws.name
    left join weeek_time wtt on wtt.employee = ws.name
    left join data_store.b_employees be on be.name = ws.name
    join calendar c on 1=1
    left join skid_counts sc on sc.name = ws.name 
    left join task_time tt on ws.name = tt.name
    left join absence a on a.name = ws.name
    left join late_min lm on lm.name = ws.name
    left join head_empls he on he.name = ws.name
        where (e.str_lvl>1 or e.name in ('Ефремов Василий Петрович', 'Кокорев Андрей Александрович','Радченко Наталья Владимировна','Фатиханов Руслан Ильдарович','Волкова Наталья Васильевна'))and
    (e.deleted = false or e.name in ('Шишков Дмитрий Дмитриевич','Брюханова Елизавета Андреевна','Немировченко Полина Геннадиевна',
'Брюханова Елизавета Андреевна','Дряев Георгий Георгиевич','Шумова Вера Васильевна',
'Кошкин Александр Игоревич','Белокопытов Дмитрий Константинович',
'Бакланов Федор Алексеевич','Леонтьева Светлана Викторовна','Ширинкин Артем Евгеньевич','Чубукова Александра Сергеевна','Михеева Анастасия Владимировна'))-- список тех кто уволен или был переведен в другой отдел последним числом квартала

    order by ws.name )
    
    -- where e.str_lvl!=0 and e.deleted = false
    -- where (e.str_lvl>1 or e.name = 'Ефремов Василий Петрович')and coalesce(vac_time,0)<work_t_base --and e.deleted = false
    -- and e.name not in ('Жмурина Ирина Владимировна','Наумова Анастасия Дмитриевна','Хорошева Екатерина Сергеевна')
    -- order by ws.name )
    select distinct *
    from 
    	(select 
        employee,
        dept,
        --zgd,
        -- zgd_email,
        case when no_stat = true then 'нет' else 'да' end as no_stat,
        round(work_time,2),
        avg_working_time as pers_work_dinner,
--        round(time_for_work,2) as time_for_work,
        late_min,
        case 
            when no_stat = true then 'не учитывается'
            when late_min<=200  then 'выполнен'
        else 'не выполнен' end as late_kpi,
        concat(pers_task,' %') as pers_task,
        case 
            when task_time <10 then 'не учитывается'
            when pers_task>=100 then 'выполнен' 
        	when pers_task>=85 then 'частично выполнен'
        	else 'не выполнен' end as bitrix_kpi,

        emp_mail
   
        
    from pivot ) p
    """
with skud_va.cursor() as cur:
    cur.execute (kpi_sql)
    kpi = pd.DataFrame (data = cur.fetchall(), columns = ['name','dept','no_stat','with_dinner','pers_work_with_dinner',
                                                            'lates_minute','lates_incident',
                                                          'pers_task',
                                                            'tasks_incident','email'])#'head_name','email'])
kpi.shape

(213, 10)

In [8]:
kpi.to_excel('C:/Users/n.ilina/Desktop/kpi 3 квартал/весь_кпи.xlsx')

In [9]:
#С Волковой согласовано списание опозданий, так как были выходы во время отпуска
kpi.loc[kpi['name']=='Попкова Валентина Александровна','lates_minute']=0
kpi.loc[kpi['name']=='Попкова Валентина Александровна','lates_incident']='выполнен'


kpi.loc[kpi['name']=='Ямщиков Игорь Вячеславович','pers_task']='112.48 %'
kpi.loc[kpi['name']=='Ямщиков Игорь Вячеславович','tasks_incident']='выполнен'

kpi.loc[kpi['name']=='Рубина Анастасия Евгеньевна','pers_task']='100.64 %'
kpi.loc[kpi['name']=='Рубина Анастасия Евгеньевна','tasks_incident']='выполнен'

kpi.loc[kpi['name']=='Афанасьева Маргарита Александровна','lates_minute']=2
kpi.loc[kpi['name']=='Афанасьева Маргарита Александровна','lates_incident']='выполнен'


In [11]:
kpi[kpi['name']=='Телюбаев Жаслан Барлыкович']

,name,dept,no_stat,with_dinner,pers_work_with_dinner,lates_minute,lates_incident,pers_task,tasks_incident,email


In [12]:
kpi[kpi['name']=='Харин Павел Олегович']

,name,dept,no_stat,with_dinner,pers_work_with_dinner,lates_minute,lates_incident,pers_task,tasks_incident,email
183,Харин Павел Олегович,Направление консалтинга,нет,448.03,99.39,0,не учитывается,91.93 %,частично выполнен,p.harin@rt-techpriemka.ru


In [13]:
kpi[kpi['name']=='Афанасьева Маргарита Александровна']

,name,dept,no_stat,with_dinner,pers_work_with_dinner,lates_minute,lates_incident,pers_task,tasks_incident,email
7,Афанасьева Маргарита Александровна,Направление стандартизации,да,470.03,118.84,2,выполнен,92.70 %,частично выполнен,m.afanasyeva@rt-techpriemka.ru


In [14]:
# Там где получилось отрицательно время работы, заменяем значения на 0
kpi.loc[kpi['with_dinner'] < 0, 'with_dinner'] = 0

In [15]:
kpi.to_csv('C:/Users/n.ilina/Desktop/kpi 3 квартал/kpi_3_25.csv')

In [16]:
wm_sql = """
with
empls as (
    select
        e."name" ,
        e.dept_id ,
        e.str_lvl ,
        d.short_name as dept,
        be.email as email,
        case 
        when d.short_name in ('Центр мониторинга и анализа качества','Направление проверки и оценки предприятий','Направление стандартизации') then 'Афанасьева Маргарита Александровна'
	        when e.name like 'Чубукова%' then 'Кошелева Елизавета Робертовна'
	        when d.short_name in ('Секретариат','ДОКР','Советник ГД') then 'Волкова Наталья Васильевна' 
            when e.name like any (array['Каменскова%','Шевяков%']) then 'Старшинов Антон Владимирович'
            when e.name like 'Пшеничный%' then 'Пшеничный Дмитрий Алексеевич' 
            when e.name like 'Тигиев%' then 'Виноградов Всеволод Владимирович' 
            when e.name like 'Трусов Михаил%' then 'Ефремов Василий Петрович' 
            when e.name like 'Денисенко%' then 'Балашов Роман Александрович'
            when e.name like any (array['Мазур%','Яковлева Елена%']) then 'Мазур Алексей Борисович' 
            when e.name like 'Клишин%' then 'Клишин Андрей Иванович'
            when d.short_name ='ДУП' then 'Радченко Наталья Владимировна'
            end as head,
        case  
        	   when d.short_name in ('Центр мониторинга и анализа качества','Направление проверки и оценки предприятий','Направление стандартизации') then 'm.afanasyeva@rt-techpriemka.ru'
        when e.name = 'Чубукова Александра Сергеевна' then 'e.kosheleva@rt-techpriemka.ru'
        when d.short_name in ('Секретариат','ДОКР','Советник ГД') then 'n.volkova@rt-techpriemka.ru'
        when e.name like any (array['Каменскова%','Шевяков%']) then 'a.starshinov@rt-techpriemka.ru'
        when e.name like 'Пшеничный%' then 'd.pshenichnyy@rt-techpriemka.ru' 
        when e.name like 'Тигиев%' then 'v.vinogradov@rt-techpriemka.ru'
        when e.name like 'Денисенко%' then 'r.balashov@rtqualityplus.ru'
        when e.name like 'Трусов Михаил%' then 'v.efremov@rtqualityplus.ru'
        when e.name like any (array['Мазур%','Яковлева Елена%']) then 'a.mazur@rt-techpriemka.ru'
        when e.name like 'Клишин%' then 'a.klishin@rt-techpriemka.ru'
        when d.short_name ='ДУП' then 'n.radchenko@rt-techpriemka.ru'
        end	as head_mail 
    from dds.employees e
    left join data_store.c_employees ce on ce.uniq_id = e.uniq_id and ce.deleted = false
    join dds.departments d on d.dept_id = e.dept_id
    join data_store.b_employees be on be.name = e.name 
    --where (str_lvl not in (1,0))
    order by d.short_name , e.str_lvl
    ),
    head_empls as (
    select 
        distinct
        coalesce(e.head,h.name) as head_name,
        coalesce (e.head_mail,h.email) as email,
        e.name as name
    from empls e  
    left join empls h on h.dept = e.dept 
    where (e.str_lvl in (2,3) and h.str_lvl =2) 
    or e.head like any (array ['%Волкова%','Пшеничный%','Старшинов%','Мазур%','Кондова%','Виноградов%','Клишин%','Кошелева%','Ефремов%'])
    )
SELECT
						distinct
					wm."date",
					wm.name,
					CASE 
						WHEN wm.name='Румянцева Анастасия Игоревна' AND wm.date='2025-04-02' THEN '09:00:00'
						ELSE wm.start_time
					END AS start_time,
					wm.end_time ,
					wm.working_time ,
					coalesce (wm.b_abc,case 
						when ca.start_date::date = wm.date::date and ca.end_date::date = wm.date::date 
						then concat (to_char(ca.start_date::time,'HH24:MM'), '-', to_char(ca.end_date::time,'HH24:MM'))
						when wm."date"::date >=ca.start_date::DATE and wm.date<=ca.end_date::DATE and ca.end_date::time>e.end_wd 
						then concat (to_char(e.start_wd ::time,'HH24:MM'), '-', to_char(e.end_wd ::time,'HH24:MM'))
					end) as time_out
					,
					ca.info  as info,
                    be.email as emp_mail
					--he.head_name,
					--he.email
				FROM dds.work_mart wm
				JOIN dds.employees e ON e."name" = wm."name" 
                left join data_store.b_employees be on be.name = wm.name
				left JOIN (SELECT name FROM data_store.c_employees ce WHERE ce.dismiss_dt isnull) ce ON ce.name = wm.name 
				LEFT JOIN 
					(SELECT 
						c.*,
						coalesce (bat.info,c.typeofabsence) as info
					FROM data_store.c_absences c left join data_store.b_absence_type bat on bat.typeofabsence =  c.typeofabsence
					) ca ON ca.cardnumber = e.tabel_number AND (wm."date"::DATE = ca.start_date::DATE or (wm."date"::date >=ca.start_date::DATE and wm.date<=ca.end_date::DATE))
				left join head_empls he on he.name = wm.name
				where wm.date between '2025-07-01' and '2025-09-30'
                and head_name not in ('Цыбулевский Кирилл Александрович','Куц Иван Александрович','Трубин Олег Андреевич')
				and head_name notnull
				order by name, date asc
 """
with skud_va.cursor() as cur:
    cur.execute (wm_sql)
    work_mart = pd.DataFrame (data = cur.fetchall(), 
                              columns = ['date','name','start_time','end_time',
                                         'working_time','time_out','info','email'])
                                         #'head_name','email'])
work_mart.shape

(12865, 8)

In [17]:
work_mart.to_csv('C:/Users/n.ilina/Desktop/kpi 3 квартал/work_mart.csv')

In [18]:
#Добавлены костыли для майских-июньских праздников
sad_sql = """
with 
dates as (
select
	'2025-07-01'::date as st, --начало периода
	'2025-09-30'::date as edt, --конец периода
	'y' as vac, --учет отпуска (y/n)
	0 as delay --смещение просрочки до конца дня если 1, если 0, то просрочка секунда в секунду
	),
tasks as (
select 
	da.task,
	da.maintask,
	da.subject,
	dr.name,
	cd."name" as dept,
	dr.email_company_sungero as email,
	da.author ,
	da.performer,
	da.created,
	extract(month from created) as month,
	da.completed,
	case 
		when da.deadline::time ='00:00:00' then da.deadline::date +'23:59:59'::time
		else da.deadline end as deadline,
	case 
		when da.deadline::time ='00:00:00' then da.completed - (da.deadline::date +'23:59:59'::time)
	else da.completed - da.deadline end as before_deadline,
	case --расчет нового деделайна, если учитываем отсутствия		
			
		when duration notnull and da.deadline::time ='00:00:00' and extract(dow from (da.deadline::date + duration * interval '1 days')) = 6 
			then da.deadline::date + (duration+2) * interval '1 days'+ '23:59:59'::time --если перенос дедлана выпадает на субботу и время дедлайна 00, + 2дня и 23 часа
		when duration notnull and da.deadline::time ='00:00:00' and extract(dow from (da.deadline::date + duration * interval '1 days')) = 0 
			then da.deadline::date + (duration+1) * interval '1 days'+ '23:59:59'::time --если перенос на воскресенье и время дедлана 00, + 1день и 23 часа
		when duration notnull and da.deadline::time ='00:00:00' --если перенос на будний и дедлайн в 00
			then da.deadline::date + (duration) * interval '1 days'+ '23:59:59'::time
		when duration notnull and extract(dow from (da.deadline::date + duration * interval '1 days')) = 6
			then da.deadline + (duration+2) * interval '1 days' --если время дедлайна задано и перенос на субботу
		when duration notnull and extract(dow from (da.deadline::date + duration * interval '1 days')) = 0
			then da.deadline + (duration+1) * interval '1 days'  --если время дедлайна задано и перенос на аоскресенье
		when duration notnull then da.deadline + (duration) * interval '1 days' --если перенос на будний и время задано
		when duration isnull and da.deadline::time ='00:00:00' then da.deadline::date +'23:59:59'::time --старый дедлайн с корректировной времени, если нет переноса и время 00
		else da.deadline end as new_deadline, --остается старый дедлайн, если нет переноса
	case --новый расчет срока выполнений задачи от дедлайна, если учитываются отсутствия
			
			
		when duration notnull and da.deadline::time ='00:00:00' and extract(dow from (da.deadline::date + duration * interval '1 days')) = 6 
			then da.completed - (da.deadline::date + (duration+2) * interval '1 days'+ '23:59:59'::time)
		when duration notnull and da.deadline::time ='00:00:00' and extract(dow from (da.deadline::date + duration * interval '1 days')) = 0
			then da.completed -(da.deadline::date + (duration+1) * interval '1 days'+ '23:59:59'::time)
		when duration notnull and extract(dow from (da.deadline::date + duration * interval '1 days')) = 6
			then da.completed - (da.deadline::date + (duration+2) * interval '1 days'+ '23:59:59'::time)
		when duration notnull and extract(dow from (da.deadline::date + duration * interval '1 days')) = 0
			then da.completed - (da.deadline::date + (duration+1) * interval '1 days'+ '23:59:59'::time)
        when duration notnull and da.deadline::time ='00:00:00' 
            then da.completed - (da.deadline::date + (duration) * interval '1 days' +'23:59:59'::time)
		when duration notnull then da.completed - (da.deadline::date + (duration) * interval '1 days')
		when duration isnull and da.deadline::time ='00:00:00' then da.completed - (da.deadline::date +'23:59:59'::time)
		else da.completed - da.deadline end as new_before_deadline,
	start_date,
	end_date,
	duration
from 
	(select 
id,
discriminator,
status ,
performer ,
completedby,
task ,
subject ,
author,
created ,
modified ,
deadline as dddde,
case when deadline::date between '2024-10-10'::date and '2024-10-13'::date  
	then '2024-10-14 23:59:59' 
	when deadline::date between '2024-11-22'::date and '2024-11-24'::date  
	then '2024-11-25 23:59:59' 
	else deadline end as deadline,
maintask,
"result" ,
completed 
	from data_store.d_assignments da
    where performer<>83) da 
    --------------------------------------------------------------completedby  performer
	join data_store.d_recipient dr on dr.id = 
		(case when dr.id in (53,597,868,1431) and (performer = completedby or da.performer in (907,908,909,1343,1345,985,1482,1344,887,1037)) 
		then da.completedby else da.performer end ) --костыль на ДОК
	left join data_store.c_employees ce on ce.name = dr.name
	left join data_store.c_departments cd on cd.id = ce.department 
	left join 
	(
		select 
			ce.name,
			ca.start_date::date ,
			ca.end_date::date,
			extract(days from(ca.end_date - ca.start_date+interval '1 days')) as duration
		from data_store.c_employees ce
		join data_store.c_absences ca on ca.cardnumber =ce.id 
		join dates on 1=1
		where 
			ca.typeofabsence in (
            'Болезнь', 'Командировка',
            'Отпуск основной',
            'Отпуск по беременности и родам',
            'Болезнь без оплаты',
            'Дополнительный отпуск',
			'Отпуск неоплачиваемый по разрешению работодателя',
            'Дополнительный отпуск',
            'Отсутствие с сохранением оплаты',
			'Дополнительные выходные дни не оплачиваемые') --список отутствий, учитываемых для изменения дедлайна
			and (ca.start_date between st and edt or ca.end_date between st and edt)
) vacations on dr.name = vacations.name and da.deadline::date between vacations.start_date and vacations.end_date and da.created::date between vacations.start_date and vacations.end_date
--	and case 
--		when da.deadline::time ='00:00:00' then da.completed - (da.deadline::date +'23:59:59'::time)
--	else da.completed - da.deadline end>interval '0 seconds'
	join dates on 1=1
where 
	ce.start_dt<='2025-07-01'
	and
	da.subject not like all (array['Важно. Напоминание о сроках исполнения поручений%','%Приняты работы:%',
	'%Аннулирован контрагентом:%','%Подписан документ:%','%Отправлен на повторное согласование:%',
	'%Срок изменен на%','%Прекращено согласование:%', 'Ознакомьтесь%','Завершите работы по ознаком%',
	'Распечатайте и передайте на подпись:%','Распечатайте:%']) --исключение технических задач '%Ознакомьтесь%', '%Срок продлен до%','%Начаты работы:%'
	and da.status<>'Aborted' 
	and (created::date between st and edt and da.deadline::date between st and edt --поставлена и дедлайн в периоде
		or completed::date between st and edt --выполнена в периоде
		or da.deadline::date between st and edt and (completed isnull or completed::date between st and edt or completed>edt) --дедлайн в периоде и задача в работе
		or (created::date<st and da.deadline::date<st and (completed isnull or completed::date between st and edt))
		) --задачи предыдущих периодов, выполненые в периоде или еще в работе
	and dr.name not in (
		select name 
		from (select name, max(coalesce(dismiss_dt, '2025-10-01':: date)) as dismiss_dt from data_store.c_employees ce group by 1 ) t
		where t.dismiss_dt < edt+interval '1 days'	) --сотрудник не уволился до окончания периода
	and da.performer not in (50,15,348, 1266) 
    --and da.performer not in (50,15,1037,348,1345,985,1482,1344,887, 1266) --убираем регистраторов и сервисных пльзователей
)  ---закончился блок с выгрузкой по задачам из СЭД


select 
	name,
	dept,
	all_tasks,
	done_tasks,
	case when vac='y' then in_time_y else in_time_n end as in_time,
	case when vac='y' then delay_y
	else delay_n end as delay,
--	case when vac ='y' then in_time_y+sed_delay_y else in_time_n +sed_delay_n end as with_sed_in_time,
--    case when vac ='y' then delay_y - sed_delay_y else delay_n - sed_delay_n end as with_sed_delay,
	round((case when vac='y' then in_time_y else in_time_n end) ::numeric/done_tasks::numeric*100,2) as pers_in_time
	--round((case when vac='y' then in_time_y+sed_delay_y else in_time_n +sed_delay_n end) ::numeric/done_tasks::numeric*100,2) as pers_in_time_with_sed
  -- round((case when vac='y' then delay_y else delay_n end)::numeric/done_tasks::numeric*100,2) as pers_delay
from (
	select 
		tasks.name,
		tasks.dept,
		count(*) as all_tasks,
		count(*) filter (where completed notnull) as done_tasks,
		count(*) filter (where before_deadline<=delay*('23:59:59'::time -deadline::time) or (completed notnull and deadline isnull)) as in_time_n,
		count(*) filter (where before_deadline>delay*('23:59:59'::time -deadline::time) ) as delay_n,
		count(*) filter (where new_before_deadline<=delay*('23:59:59'::time -new_deadline::time) or (completed notnull and deadline isnull)) as in_time_y,
		count(*) filter (where new_before_deadline>delay*('23:59:59'::time -new_deadline::time) ) as delay_y,
		count(*) filter (where new_before_deadline>delay*('23:59:59'::time -new_deadline::time) 
		and new_deadline::date between '2024-11-22' and '2024-11-24' and completed::date <='2024-11-25') as sed_delay_y,
		count(*) filter (where before_deadline>delay*('23:59:59'::time -deadline::time) 
		and deadline::date between '2024-11-22' and '2024-11-24' and completed::date ='2024-11-25') as sed_delay_n,
		vac
	from tasks
	join dates on 1=1
	group by  tasks.name, tasks.dept,vac
	having count(*) filter (where completed notnull)>0
	) p
ORDER BY pers_in_time
	
"""

with skud_va.cursor() as cur:
    cur.execute (sad_sql)
    sed_stat = pd.DataFrame (data = cur.fetchall(), 
                             columns = ['name','dept','all_tasks','done_tasks',
                                        'with_sed_in_time','with_sed_delay','pers_in_time'])

print(sed_stat.shape)
skud_va.close()

(384, 7)


In [19]:
# Осокина Наталья Геннадьевна 2 задачи ошибочно просрочены 
sed_stat.loc[sed_stat['name']=='Осокина Наталья Геннадьевна','with_sed_in_time']=139
sed_stat.loc[sed_stat['name']=='Осокина Наталья Геннадьевна','with_sed_delay']=0
sed_stat.loc[sed_stat['name']=='Осокина Наталья Геннадьевна','pers_in_time']=100.00

#Привезенцева Марина Николаевна 1 задача ошибочно просрочена

sed_stat.loc[sed_stat['name']=='Привезенцева Марина Николаевна','with_sed_in_time']=11
sed_stat.loc[sed_stat['name']=='Привезенцева Марина Николаевна','with_sed_delay']=0
sed_stat.loc[sed_stat['name']=='Привезенцева Марина Николаевна','pers_in_time']=100.00

#Мезенцева Оксана Васильевна 2 задачи ошибочно просрочены 

sed_stat.loc[sed_stat['name']=='Мезенцева Оксана Васильевна','with_sed_in_time']=268
sed_stat.loc[sed_stat['name']=='Мезенцева Оксана Васильевна','with_sed_delay']=0
sed_stat.loc[sed_stat['name']=='Мезенцева Оксана Васильевна','pers_in_time']=100.00

#Лапкина Елена Ивановна Васильевна 2 задачи ошибочно просрочены 

sed_stat.loc[sed_stat['name']=='Лапкина Елена Ивановна','with_sed_in_time']=725
sed_stat.loc[sed_stat['name']=='Лапкина Елена Ивановна','with_sed_delay']=8
sed_stat.loc[sed_stat['name']=='Лапкина Елена Ивановна','pers_in_time']=98.91


#Гайнуллина Лилия Ильдусовна 1 задача ошибочно просрочены 

sed_stat.loc[sed_stat['name']=='Гайнуллина Лилия Ильдусовна','with_sed_in_time']=215
sed_stat.loc[sed_stat['name']=='Гайнуллина Лилия Ильдусовна','with_sed_delay']=0
sed_stat.loc[sed_stat['name']=='Гайнуллина Лилия Ильдусовна','pers_in_time']=100.00


#Попкова 2 задачи ошибочно просрочены 

sed_stat.loc[sed_stat['name']=='Попкова Валентина Александровна','with_sed_in_time']=38
sed_stat.loc[sed_stat['name']=='Попкова Валентина Александровна','with_sed_delay']=0
sed_stat.loc[sed_stat['name']=='Попкова Валентина Александровна','pers_in_time']=100.00


# на задачу было выделено всего 3 минуты Яковлева Юлия Николаевна

sed_stat.loc[sed_stat['name']=='Яковлева Юлия Николаевна','with_sed_in_time']=666
sed_stat.loc[sed_stat['name']=='Яковлева Юлия Николаевна','with_sed_delay']=0
sed_stat.loc[sed_stat['name']=='Яковлева Юлия Николаевна','pers_in_time']=100.00

In [20]:
sed_stat[sed_stat['name']=='Яковлева Юлия Николаевна']

,name,dept,all_tasks,done_tasks,with_sed_in_time,with_sed_delay,pers_in_time
244,Яковлева Юлия Николаевна,АУП 06.2: Главный бухгалтер (сч 26),666,666,666,0,100.0


In [21]:
sed_stat[sed_stat['name']=='Шумова Вера Васильевна']

,name,dept,all_tasks,done_tasks,with_sed_in_time,with_sed_delay,pers_in_time
44,Шумова Вера Васильевна,1.3.2 Направление сертификации оборонно-промыш...,112,112,88,24,78.57
45,Шумова Вера Васильевна,"АУП 01.4: ЦС ""Ростех-сертификат"" (20 сч)",112,112,88,24,78.57


In [23]:
sed_stat[sed_stat['name']=='Телюбаев Жаслан Барлыкович']

,name,dept,all_tasks,done_tasks,with_sed_in_time,with_sed_delay,pers_in_time
356,Телюбаев Жаслан Барлыкович,АУП 02.2.1: Направление реализации проектов по...,64,64,64,0,100.00


In [22]:
sed_stat.to_csv('C:/Users/n.ilina/Desktop/kpi 3 квартал/sed_stat.csv')

In [35]:
kpi = kpi.merge(sed_stat.fillna(0), how='right', on ='name')

In [36]:
kpi['dept_x'] = kpi['dept_x'].fillna(kpi['dept_y'])

In [37]:
kpi = kpi.fillna('-')

In [38]:
kpi['pers_in_time'] =kpi['pers_in_time'].map(lambda x: str(x)+' %' )
kpi['pers_work_with_dinner'] =kpi['pers_work_with_dinner'].map(lambda x: str(x)+' %' if x!='-' else '-')

In [39]:
kpi = kpi.drop(columns = 'dept_y')

In [40]:
kpi = kpi.drop(columns = 'dept_x')

In [41]:
kpi = kpi.drop_duplicates()

In [42]:
kpi.to_csv('kpi3_25.csv')

In [43]:
# # #Закоментировать если нужно сформировать файлы по каждому руководителю
# work_mart['head_name'] ="Кондратенко Денис Анатольевич" 
# kpi['head_name'] ="Кондратенко Денис Анатольевич" 
# #используется для тестирования рассылки
# work_mart['email'] ="d.kondratenko@rt-techpriemka.ru" 
# kpi['email'] ="d.kondratenko@rt-techpriemka.ru" 
# kpi['email'] ='n.ilina@rt-techpriemka.ru'
# kpi['head_name'] ='Ильина Наталья Павловна'
# work_mart['email'] ='n.ilina@rt-techpriemka.ru'
# work_mart['head_name'] ='Ильина Наталья Павловна'

In [44]:
def create_file():
    for addr_to in kpi['email'].unique():
        i = kpi.loc[kpi['email']==addr_to,'name'].unique()[0]
        #file_name = 'C:/Users/n.ilina/Desktop/KPI 2 кв/По каждому сотруднику/ {}.xlsx'.format(i)
        file_name = 'C:/Users/n.ilina/Desktop/kpi 3 квартал/По каждому сотруднику/ {}.xlsx'.format(i)
        print(i)
        if os.path.isfile(file_name):
            os.remove(file_name)
        work_mart_send = work_mart.loc[work_mart['email'] == addr_to]
        work_mart_send = work_mart_send.drop(['email'], axis = 1)
        dept_kpi = kpi.loc[kpi['email'] == addr_to]
        dept_kpi = dept_kpi.drop(['email'], axis = 1)
        dept_kpi = dept_kpi.rename(columns = {'name':'ФИО работника',
                                              'no_stat':'Учет в статистике',
                                              'with_dinner': 'Отработано, ч.',
                                              'pers_work_with_dinner':'% отработанного времени',
                                              'lates_minute':'Опоздания (минуты)',
                                              'lates_incident':'Выполнение КПЭ по опозданиям',
                                              'pers_task':'Процент списания времени Б24/weeek',
                                             'tasks_incident':'Выполнение КПЭ по списанию времени Б24/weeek',    
                                              'all_tasks':'Всего задач',
                                              'done_tasks': 'Выполнено задач в СЭД',
                                              'with_sed_in_time':'Выполнено в срок',
                                              'with_sed_delay':'Выполено с просросчкой',
                                              'pers_in_time':'% выполненых в срок СЭД'
                                             })
        work_mart_send = work_mart_send.rename(columns = {'date':'Дата',
                                              'name':'ФИО работника',
                                             'start_time':'Начало рабочего дня',
                                             'end_time':'Окончание рабочего дня',
                                             'working_time':'Отработанное время',
                                            'time_out':'Время отсутствия',
                                            'info':'Причина отстутствия'
                                             })
        if dept_kpi['Процент списания времени Б24/weeek'].unique().all()== '0 %':
            dept_kpi = dept_kpi.drop([
    #             'Время на задачи, часов','Время, списанное на задачи',
                                      'Процент списания времени Б24/weeek',
                                      'Выполнение КПЭ по списанию времени Б24/weeek'], axis = 1)

        with pd.ExcelWriter(file_name) as writer:
            dept_kpi.to_excel(writer,index =False, sheet_name='КПЭ')
            work_mart_send.to_excel(writer,index =False, sheet_name='учет_рабочего_времени')
        wb = openpyxl.load_workbook(file_name)
        ws = wb['КПЭ']   
        font_size = 11
        ws.column_dimensions['A'].width = 35
        ws.column_dimensions['B'].width = 15
        ws.column_dimensions['C'].width = 12
        ws.column_dimensions['D'].width = 14
        ws.column_dimensions['E'].width = 15
        ws.column_dimensions['F'].width = 15
        ws.column_dimensions['G'].width = 15
        ws.column_dimensions['H'].width = 15
        ws.column_dimensions['I'].width = 15
        ws.column_dimensions['J'].width = 15
        ws.column_dimensions['K'].width = 15
        ws.column_dimensions['L'].width = 15
        ws.column_dimensions['M'].width = 15
        ws.column_dimensions['N'].width = 15
        comment = Comment('Из рабочего времени исключены согласованные отсутствия, кроме удаленной работы', 
                          'Comment Author')
        comment.width = 300
        comment.height = 80
        if ws['E1'].value !='':
            ws["E1"].comment = comment
        set_table(ws,'A1:'+openpyxl.utils.get_column_letter(ws.max_column)+str(ws.max_row))
        set_style(ws, 'A1:'+openpyxl.utils.get_column_letter(ws.max_column)+str(1))    
        ws = wb['учет_рабочего_времени']  
        ws.column_dimensions['A'].width = 12
        ws.column_dimensions['B'].width = 23
        ws.column_dimensions['C'].width = 13
        ws.column_dimensions['D'].width = 13
        ws.column_dimensions['E'].width = 14
        ws.column_dimensions['F'].width = 15
        ws.column_dimensions['G'].width = 18
        set_table(ws,'A1:G'+str(ws.max_row))
        set_style(ws, 'A1:G1')   
        maxcolumnletter = openpyxl.utils.get_column_letter(ws.max_column)
        ws.auto_filter.ref = 'A1:'+maxcolumnletter+str(len(ws['A']))
        wb.save(file_name)

In [45]:
create_file()

Букина Инна Васильевна
Алескеров Игорь Тимурович
Синев Алексей Петрович
Лященко Леонид Алексеевич
Агапов Владимир Владимирович
Керимов Денис Евгеньевич
Михальчук Ксения Александровна
Яненко Игорь Юрьевич
Ветрова Наталья Андреевна
Баскаков Ярослав Алексеевич
Чибисов Олег Владимирович
Чураков Илья Михайлович
Тулецков Виктор Владимирович
Диденко Тамара Андреевна
Бугакова Ирина Павловна
Мухаев Егор Владимирович
Запорожский Вячеслав Александрович
Володичев Артем Николаевич
Батуркин Максим Анатольевич
Кошелева Елизавета Робертовна
Сайбель Тимофей Александрович
Квасова Софья Сергеевна
Палицын Антон Сергеевич
Ковальчук Сергей Петрович
Афанасьева Маргарита Александровна
Кириллова Елена Владиславовна
Фомин Павел Александрович
Клячкина Екатерина Александровна
Виноградова Надежда Сергеевна
Колесников Вадим Сергеевич
Камолова Анастасия Андреевна
Шумова Вера Васильевна
Мазурин Александр Алексеевич
Батурин Александр Васильевич
Шигин Семен Андреевич
Белкин Григорий Викторович
Кутьев Анатолий Павлович


In [61]:
#убираем пользователей, по которым не учитывается статистика 
#kpi=kpi[kpi['no_stat']=='да'].reset_index(drop=True)

In [62]:
#удаляем не нужные колонки 
kpi=kpi.drop(columns=['no_stat','with_dinner'])

In [63]:
#функция для формирования файлов и отправки
def mes_send (addr_to):
    # for addr_to in kpi['email'].unique():
    #     i = kpi.loc[kpi['email']==mes_send('n.ilina@rt-techpriemka.ru'),'name'].unique()[0]
        i=addr_to
        file_name =  'C:/Users/n.ilina/Desktop/kpi 3 квартал/По каждому сотруднику/ {}.xlsx'.format(i)
        if os.path.isfile(file_name):
            os.remove(file_name)
        work_mart_send = work_mart.loc[work_mart['email'] == addr_to]
        work_mart_send = work_mart_send.drop(['email'], axis = 1)
        dept_kpi = kpi.loc[kpi['email'] == addr_to]
        dept_kpi = dept_kpi.drop(['email'], axis = 1)


        dept_kpi=dept_kpi
        dept_kpi = dept_kpi.rename(columns = {'name':'ФИО работника',
                                              #'no_stat':'Учет в статистике',
                                              #'with_dinner': 'Отработано, ч.',
                                              'pers_work_with_dinner':'% отработанного времени',
                                              'lates_minute':'Опоздания (минуты)',
                                              'lates_incident':'Выполнение КПЭ по опозданиям',
                                              'pers_task':'Процент списания времени Б24',
                                             'tasks_incident':'Выполнение КПЭ по списанию времени Б24',    
                                              'all_tasks':'Всего задач',
                                              'done_tasks': 'Выполнено задач в СЭД',
                                              'with_sed_in_time':'Выполнено в срок',
                                              'with_sed_delay':'Выполено с просрочкой',
                                              'pers_in_time':'% выполненых в срок СЭД'
                                             })
        work_mart_send = work_mart_send.rename(columns = {'date':'Дата',
                                              'name':'ФИО работника',
                                             'start_time':'Начало рабочего дня',
                                             'end_time':'Окончание рабочего дня',
                                             'working_time':'Отработанное время',
                                            'time_out':'Время отсутствия',
                                            'info':'Причина отстутствия'
                                             })
        if dept_kpi['Процент списания времени Б24'].unique().all()== '0 %':
            dept_kpi = dept_kpi.drop([
    #             'Время на задачи, часов','Время, списанное на задачи',
                                      'Процент списания времени Б24',
                                      'Выполнение КПЭ по списанию времени Б24'], axis = 1)

        with pd.ExcelWriter(file_name) as writer:
            dept_kpi.to_excel(writer,index =False, sheet_name='КПЭ')
            work_mart_send.to_excel(writer,index =False, sheet_name='учет_рабочего_времени')
        wb = openpyxl.load_workbook(file_name)
        ws = wb['КПЭ']   
        font_size = 11
        ws.column_dimensions['A'].width = 35
        ws.column_dimensions['B'].width = 15
        ws.column_dimensions['C'].width = 15
        ws.column_dimensions['D'].width = 14
        ws.column_dimensions['E'].width = 15
        ws.column_dimensions['F'].width = 15
        ws.column_dimensions['G'].width = 15
        ws.column_dimensions['H'].width = 15
        ws.column_dimensions['I'].width = 15
        ws.column_dimensions['J'].width = 15
        ws.column_dimensions['K'].width = 15
        ws.column_dimensions['L'].width = 15
        ws.column_dimensions['M'].width = 15
        ws.column_dimensions['N'].width = 15
        comment = Comment('Из рабочего времени исключены согласованные отсутствия, кроме удаленной работы', 
                          'Comment Author')
        comment.width = 300
        comment.height = 80
        if ws['E1'].value !='':
            ws["E1"].comment = comment
        set_table(ws,'A1:'+openpyxl.utils.get_column_letter(ws.max_column)+str(ws.max_row))
        set_style(ws, 'A1:'+openpyxl.utils.get_column_letter(ws.max_column)+str(1))    
        ws = wb['учет_рабочего_времени']  
        ws.column_dimensions['A'].width = 12
        ws.column_dimensions['B'].width = 23
        ws.column_dimensions['C'].width = 13
        ws.column_dimensions['D'].width = 13
        ws.column_dimensions['E'].width = 14
        ws.column_dimensions['F'].width = 15
        ws.column_dimensions['G'].width = 18
        set_table(ws,'A1:G'+str(ws.max_row))
        set_style(ws, 'A1:G1')   
        maxcolumnletter = openpyxl.utils.get_column_letter(ws.max_column)
        ws.auto_filter.ref = 'A1:'+maxcolumnletter+str(len(ws['A']))
        wb.save(file_name)
    
        addr_from = "no-reply@rt-techpriemka.ru"
        password  = "whtzFzZ5gpvTjvDYetiy"
#     addr_from = "d.kondratenko@rt-techpriemka.ru"     
#     password  = "SAy1kehPc0FC0EjnUGyj" 
        msg_subj = 'Результаты КПЭ за 3 квартал 2025 г.'
        msg_text =''
        msg_text = """
            <html>
            <p>Добрый день!<br /><br />
            Для формирования отчета по бонусным картам за 3 квартал 2025 года, прошу <b>проверить</b> данные, полученные из автоматизированных систем.<br/>
            <br/>
            На листе "Учет_рабочего_времени" есть подробные данные о начале и окончании рабочего дня, о чистом отработанном времени (времени нахождения внутри офиса), времени и причинах согласованных отсутствий.<br/>
            <br/>При необходимости внесения правок по показателям, которые учитываются в Вашей бонусной карте, отправьте информацию на почту n.ilina@rt-techpriemka.ru до {} с описанием даты и типом корректировки.
             <br />
             <br/>
            Спасибо! Хорошего дня!<br/>
                <br />
                <br /><br />
                </div>
    
        </html>
        """.format("{:02d}".format(corr_date.day)+'.'+"{:02d}".format(corr_date.month)+'.'+"{:04d}".format(corr_date.year)+' года')
       
        # Compose message  
        msg = MIMEMultipart()
        msg['From'] = addr_from
        msg['To'] = addr_to
        msg['Subject'] = msg_subj
        msg.attach(MIMEText(msg_text, 'html'))
        
        att1 = MIMEApplication(open(file_name,'rb').read()) 
        att1.add_header('Content-Disposition', 'attachment', subtype='xlsx',filename='Статистика KPI 3кв25.xlsx') 
        msg.attach(att1)
            
        # Send mail
        server = smtplib.SMTP_SSL('94.100.180.160', 465)
        server.login(addr_from, password)  
        server.sendmail(addr_from , addr_to, msg.as_string())
        server.quit()

In [64]:
ulist = []
ulist = list(kpi['email'].unique())
# for x in [0, 'l.ivshina@rtqualityplus.ru', 'v.smirnova@rt-techpriemka.ru']:
#     ulist.remove(x)

In [65]:
ulist

['-',
 'i.bukina@rt-techpriemka.ru',
 'i.aleskerov@rttechpostavka.ru',
 'l.lyashchenko@rt-techpriemka.ru',
 'v.agapov@rt-techpriemka.ru',
 'd.kerimov@rt-techpriemka.ru',
 'k.mihalchuk@rtqualityplus.ru',
 'i.yanenko@rtqualityplus.ru',
 'n.vetrova@rt-techpriemka.ru',
 'ya.baskakov@rttechpostavka.ru',
 'o.chibisov@rt-techpriemka.ru',
 'i.churakov@rt-techpriemka.ru',
 'v.tuleckov@rttechpostavka.ru',
 't.didenko@rt-globalconsult.ru',
 'i.bugakova@rttechpostavka.ru',
 'e.muhaev@rt-techpriemka.ru',
 'v.zaporozhskiy@rt-techpriemka.ru',
 'a.volodichev@rttechpostavka.ru',
 'm.baturkin@rt-techpriemka.ru',
 'e.kosheleva@rt-techpriemka.ru',
 't.saybel@rt-techpriemka.ru',
 's.kvasova@rttechpostavka.ru',
 'a.palicyn@rt-techpriemka.ru',
 's.kovalchuk@rt-techpriemka.ru',
 'm.afanasyeva@rt-techpriemka.ru',
 'e.kirillova@rt-techpriemka.ru',
 'p.fomin@rt-techpriemka.ru',
 'e.klyachkina@rt-techpriemka.ru',
 'n.vinogradova@rt-techpriemka.ru',
 'vs.kolesnikov@rt-techpriemka.ru',
 'v.shumova@rt-techpriemka.ru

In [66]:
# for i in range(len(ulist)):
#     try:
#         mes_send(ulist[i])
#         print ('Letter was sent', ulist[i])
#     except:
#         print ('Сообщение не отправлено по адресу: ', ulist[i])

In [39]:
#mes_send('n.ilina@rt-techpriemka.ru')

In [40]:
# mes_send('d.kondratenko@rt-techpriemka.ru')